In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

In [ ]:
tickers = yf.Tickers('CBA.AX BHP.AX CSL.AX WES.AX MQG.AX')
dt_tickers = ['CBA.AX', 'BHP.AX', 'CSL.AX', 'WES.AX', 'MQG.AX']
data = yf.download(dt_tickers, start="2021-01-01", end="2025-12-31")

In [ ]:
info = tickers.tickers['CBA.AX'].info
print(info.keys())

for ticker, obj in tickers.tickers.items():
    info = obj.info
    print(f"{ticker}: {info['longName']} | Sector: {info['sector']} | Market Cap: {info['marketCap']}")


In [ ]:
close_prices = data['Close']
close_prices.head()

In [ ]:
returns = close_prices.pct_change().dropna()
returns.head()

In [ ]:
mean_returns = returns.mean()
cov_matrix = returns.cov()

print("Mean Daily Returns:")
print(mean_returns)
print("\nCovariance Matrix:")
print(cov_matrix)


In [ ]:
# Monte Carlo simulation parameters
num_portfolios = 10000
num_stocks = len(mean_returns)
trading_days = 252

# Fixed random seed so the results are the same every time the notebook runs
rng = np.random.default_rng(42)

results = np.zeros((3, num_portfolios))
weights_record = []

for i in range(num_portfolios):
    # Randomly generate portfolio weights
    weights = rng.random(num_stocks)
    # Normalise weights to sum to 1
    weights /= np.sum(weights)
    # Save the combination of weights for later analysis
    weights_record.append(weights)

    # Calculate portfolio return in annualised terms
    portfolio_return = np.sum(mean_returns * weights) * trading_days

    # Calculate portfolio volatility
    port_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix * trading_days, weights)))

    # Sharpe ratio (assuming risk-free rate of 4% for Australia)
    sharpe_ratio = (portfolio_return - 0.04) / port_volatility

    # Results
    results[0, i] = portfolio_return
    results[1, i] = port_volatility
    results[2, i] = sharpe_ratio

print(results.shape)
print(results[:, :5])  # show first 5 portfolios

In [ ]:
# Plotting the result.
plt.figure(figsize=(12, 8))
plt.scatter(results[1], results[0], c=results[2], cmap='viridis', alpha=0.5, s=10)
plt.colorbar(label='Sharpe Ratio')
plt.xlabel('Volatility (Risk)')
plt.ylabel('Annual Return')
plt.title('Efficient Frontier - ASX Portfolio')
plt.show()

In [ ]:
# Maximum Sharpe Ratio Portfolio
max_sharpe_idx = np.argmax(results[2])
max_sharpe_return = results[0, max_sharpe_idx]
max_sharpe_volatility = results[1, max_sharpe_idx]
max_sharpe_ratio = results[2, max_sharpe_idx]
max_sharpe_weights = weights_record[max_sharpe_idx]

print("Max Sharpe Ratio Portfolio:")
print(f"Return: {max_sharpe_return:.4f}")
print(f"Volatility: {max_sharpe_volatility:.4f}")
print(f"Sharpe Ratio: {max_sharpe_ratio:.4f}")
print("Weights:")
print(pd.Series(max_sharpe_weights, index=mean_returns.index).round(4).to_string())

In [ ]:
# Minimum Variance Portfolio
min_var_idx = np.argmin(results[1])
min_var_return = results[0, min_var_idx]
min_var_volatility = results[1, min_var_idx]
min_var_sharpe = results[2, min_var_idx]
min_var_weights = weights_record[min_var_idx]

print("Minimum Variance Portfolio:")
print(f"Return: {min_var_return:.4f}")
print(f"Volatility: {min_var_volatility:.4f}")
print(f"Sharpe Ratio: {min_var_sharpe:.4f}")
print("Weights:")
print(pd.Series(min_var_weights, index=mean_returns.index).round(4).to_string())

In [ ]:
plt.figure(figsize=(12, 8))

# plot all portfolios
plt.scatter(results[1], results[0], c=results[2], cmap='viridis', alpha=0.5, s=10)
plt.colorbar(label='Sharpe Ratio')

# mark max sharpe portfolio
plt.scatter(max_sharpe_volatility, max_sharpe_return, marker='*', color='red', s=300, label='Max Sharpe Ratio')

# mark minimum variance portfolio
plt.scatter(min_var_volatility, min_var_return, marker='*', color='blue', s=300, label='Min Variance')

plt.xlabel('Volatility (Risk)')
plt.ylabel('Annual Return')
plt.title('Efficient Frontier - ASX Portfolio')
plt.legend()

# Save the chart so the README can show it
Path('../images').mkdir(exist_ok=True)
plt.savefig('../images/efficient_frontier.png', bbox_inches='tight')
plt.show()